# 🏏 IPL Player Performance - ML Model
### Regression + Classification on IPL 2025 Player Lifetime Statistics

**What we will predict:**
- 🔵 **Regression** → Predict how many Runs a player will score
- 🟢 **Classification** → Predict if a player is an All-Rounder (Yes/No)

---

## Step 1 — Import Libraries
These are the tools we need. Think of them like a toolbox.

In [ ]:
import pandas as pd               # for loading and handling data
import numpy as np                # for numbers and math
import matplotlib.pyplot as plt   # for drawing charts
import seaborn as sns             # for prettier charts

from sklearn.model_selection import train_test_split   # split data into train/test
from sklearn.preprocessing import StandardScaler       # scale/normalize features
from sklearn.linear_model import LinearRegression      # regression model
from sklearn.ensemble import RandomForestClassifier    # classification model
from sklearn.metrics import (
    mean_absolute_error, r2_score,                     # regression metrics
    accuracy_score, classification_report,             # classification metrics
    confusion_matrix                                   # confusion matrix
)

import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

## Step 2 — Load the Dataset
Put your CSV file in the same folder as this notebook and update the filename below.

In [ ]:
# ⚠️ Update this path to where your CSV file is located
df = pd.read_csv('cricket_data_2025.csv')

print(f'✅ Dataset loaded!')
print(f'📊 Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'\n📋 Column names:')
print(df.columns.tolist())

## Step 3 — Explore the Data (EDA)
Before building any model, always look at your data first!

In [ ]:
# First 5 rows
print('🔍 First 5 rows:')
df.head()

In [ ]:
# Basic info - data types, null values
print('📊 Dataset Info:')
df.info()

In [ ]:
# Summary statistics
print('📈 Summary Statistics:')
df.describe()

In [ ]:
# Check missing values
print('❓ Missing values per column:')
print(df.isnull().sum())

In [ ]:
# Visualise Runs Scored distribution
plt.figure(figsize=(10, 4))
sns.histplot(df['Runs_Scored'], bins=30, color='steelblue', kde=True)
plt.title('Distribution of Runs Scored')
plt.xlabel('Runs Scored')
plt.ylabel('Number of Players')
plt.tight_layout()
plt.show()
print('📊 This chart shows how runs are distributed across all players')

In [ ]:
# Top 10 run scorers
top_batters = df.groupby('Player_Name')['Runs_Scored'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 5))
sns.barplot(x=top_batters.values, y=top_batters.index, palette='Blues_r')
plt.title('Top 10 Run Scorers in IPL')
plt.xlabel('Total Runs')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap - which columns relate to each other
numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(14, 10))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.1f', cmap='coolwarm', center=0)
plt.title('Correlation Between All Features')
plt.tight_layout()
plt.show()
print('🔥 Red = strong positive correlation, Blue = negative correlation')

## Step 4 — Data Cleaning & Preprocessing
Clean the data before feeding it to any model.

In [ ]:
# Replace 'No stats' with NaN, then fill with 0
df.replace('No stats', np.nan, inplace=True)

# Convert object columns to numeric where possible
for col in df.columns:
    if df[col].dtype == 'object' and col != 'Player_Name':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill remaining NaN with 0
df.fillna(0, inplace=True)

print('✅ Data cleaned!')
print(f'Missing values remaining: {df.isnull().sum().sum()}')

---
# 🔵 MODEL 1 — REGRESSION
### Goal: Predict how many Runs a player will score
This is a regression problem because Runs is a continuous number (0, 50, 250, 1000...)

In [ ]:
# Select features (inputs) for regression
regression_features = [
    'Matches_Batted',
    'Balls_Faced',
    'Batting_Strike_Rate',
    'Half_Centuries',
    'Centuries',
    'Fours',
    'Sixes',
    'Not_Outs'
]

# Target (what we want to predict)
regression_target = 'Runs_Scored'

X_reg = df[regression_features]
y_reg = df[regression_target]

print(f'✅ Features shape: {X_reg.shape}')
print(f'✅ Target shape: {y_reg.shape}')

In [ ]:
# Split into training (80%) and testing (20%)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f'Training set: {X_train_r.shape[0]} players')
print(f'Testing set : {X_test_r.shape[0]} players')
print('\n💡 We train on 80% and test on the remaining 20% the model has never seen')

In [ ]:
# Train the Linear Regression model
reg_model = LinearRegression()
reg_model.fit(X_train_r, y_train_r)

print('✅ Regression model trained!')

In [ ]:
# Predict on test set
y_pred_r = reg_model.predict(X_test_r)

# Evaluate
mae = mean_absolute_error(y_test_r, y_pred_r)
r2  = r2_score(y_test_r, y_pred_r)

print('📊 REGRESSION RESULTS')
print('=' * 35)
print(f'Mean Absolute Error (MAE) : {mae:.2f} runs')
print(f'R² Score                  : {r2:.4f}')
print()
print('💡 What these mean:')
print(f'  MAE = on average, our prediction is off by {mae:.0f} runs')
print(f'  R²  = our model explains {r2*100:.1f}% of the variation in runs')
print(f'  R² closer to 1.0 = better model')

In [ ]:
# Visualise: Actual vs Predicted runs
plt.figure(figsize=(8, 6))
plt.scatter(y_test_r, y_pred_r, alpha=0.5, color='steelblue')
plt.plot([y_test_r.min(), y_test_r.max()],
         [y_test_r.min(), y_test_r.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Runs')
plt.ylabel('Predicted Runs')
plt.title('Regression: Actual vs Predicted Runs')
plt.legend()
plt.tight_layout()
plt.show()
print('💡 Points closer to the red line = better predictions')

In [ ]:
# Feature importance for regression
coef_df = pd.DataFrame({
    'Feature': regression_features,
    'Coefficient': reg_model.coef_
}).sort_values('Coefficient', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=coef_df, x='Coefficient', y='Feature', palette='coolwarm')
plt.title('Which features influence Runs Scored the most?')
plt.tight_layout()
plt.show()
print('💡 Larger bar = more influence on the prediction')

---
# 🟢 MODEL 2 — CLASSIFICATION
### Goal: Predict if a player is an All-Rounder (Yes/No)
An All-Rounder = someone who both bats AND bowls significantly.
This is a classification problem because the answer is Yes or No (0 or 1).

In [ ]:
# Create the All-Rounder label
# Logic: player scored 100+ runs AND took 5+ wickets = All-Rounder
df['Is_AllRounder'] = ((df['Runs_Scored'] >= 100) & (df['Wickets_Taken'] >= 5)).astype(int)

all_rounder_count = df['Is_AllRounder'].sum()
non_all_rounder   = len(df) - all_rounder_count

print(f'✅ Label created!')
print(f'All-Rounders     : {all_rounder_count}')
print(f'Non All-Rounders : {non_all_rounder}')

# Visualise class balance
plt.figure(figsize=(6, 4))
sns.countplot(x='Is_AllRounder', data=df, palette=['steelblue', 'coral'])
plt.xticks([0, 1], ['Not All-Rounder', 'All-Rounder'])
plt.title('Class Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Select features for classification
classification_features = [
    'Matches_Batted',
    'Runs_Scored',
    'Batting_Strike_Rate',
    'Matches_Bowled',
    'Wickets_Taken',
    'Economy_Rate',
    'Fours',
    'Sixes',
    'Half_Centuries'
]

X_cls = df[classification_features]
y_cls = df['Is_AllRounder']

# Split data
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42
)

print(f'Training set: {X_train_c.shape[0]} players')
print(f'Testing set : {X_test_c.shape[0]} players')

In [ ]:
# Scale features - important for many ML models
scaler = StandardScaler()
X_train_c = scaler.fit_transform(X_train_c)
X_test_c  = scaler.transform(X_test_c)

print('✅ Features scaled!')

In [ ]:
# Train Random Forest Classifier
clf_model = RandomForestClassifier(n_estimators=100, random_state=42)
clf_model.fit(X_train_c, y_train_c)

print('✅ Classification model trained!')

In [ ]:
# Predict and evaluate
y_pred_c = clf_model.predict(X_test_c)

accuracy = accuracy_score(y_test_c, y_pred_c)

print('📊 CLASSIFICATION RESULTS')
print('=' * 35)
print(f'Accuracy: {accuracy * 100:.2f}%')
print()
print('Detailed Report:')
print(classification_report(y_test_c, y_pred_c, target_names=['Not All-Rounder', 'All-Rounder']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test_c, y_pred_c)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not All-Rounder', 'All-Rounder'],
            yticklabels=['Not All-Rounder', 'All-Rounder'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()
print('💡 Diagonal = correct predictions, Off-diagonal = mistakes')

In [ ]:
# Feature importance for classification
feat_imp = pd.DataFrame({
    'Feature': classification_features,
    'Importance': clf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=feat_imp, x='Importance', y='Feature', palette='Greens_r')
plt.title('Which features matter most in predicting All-Rounders?')
plt.tight_layout()
plt.show()

---
## Step 5 — Summary
Let's print a final summary of both models.

In [ ]:
print('=' * 45)
print('        🏏 IPL ML MODEL SUMMARY')
print('=' * 45)
print()
print('🔵 REGRESSION (Predicting Runs Scored)')
print(f'   MAE (avg error) : {mae:.1f} runs')
print(f'   R² Score        : {r2:.4f}')
print()
print('🟢 CLASSIFICATION (Predicting All-Rounder)')
print(f'   Accuracy        : {accuracy * 100:.2f}%')
print()
print('=' * 45)
print()
print('💡 Next steps to improve accuracy:')
print('  1. Try more features (feature engineering)')
print('  2. Try XGBoost or GradientBoostingClassifier')
print('  3. Handle class imbalance with SMOTE')
print('  4. Tune hyperparameters with GridSearchCV')